# Agilent B1500 Semiconductor Parameter Analyzer

### Initialization of the Instrument

In [ ]:
from pymeasure.instruments.agilent import AgilentB1500

# explicitly define r/w terminations; set sufficiently large timeout in milliseconds or None.
b1500 = AgilentB1500("GPIB2::17::INSTR", read_termination='\r\n', write_termination='\r\n', timeout=600000)
# query SMU config from instrument and initialize all SMU instances
b1500.initialize_all_smus()
# set data output format (required!)
b1500.data_format(21, mode=1) #call after SMUs are initialized to get names for the channels

- `\r` : *carriage return* character. move the cursor to the start of the line.
- `\n` : *line feed* character. newline.

### IV measurement with 4 SMUs

In [ ]:
# choose measurement mode
b1500.meas_mode('STAIRCASE_SWEEP', *b1500.smu_references) #order in smu_references determines order of measurement

*Set Measurement mode* of channels
- mode (`MeasMode`)
    - Spot : 단일 전압/전류 소스 -> 단일 전류/전압 측정
    - Staircase Sweep : 선형/로그 형태로 변화하는 소스 -> 각 단계에서 측정을 수행 (I-V curve 등에 사용)
    - Sampling : 지정된 시간 간격으로 연속적인 측정을 수행 (시간에 따른 전압/전류의 변화를 모니터링, transient 분석)
- arge (`SMU`) : SMU references

In [ ]:
# settings for individual SMUs
for smu in b1500.smu_references:
    smu.enable() #enable SMU
    smu.adc_type = 'HRADC' #set ADC to high-resoultion ADC
    smu.meas_range_current = '1 uA'
    smu.meas_op_mode = 'COMPLIANCE_SIDE' # other choices: Current, Voltage, FORCE_SIDE, COMPLIANCE_AND_FORCE_SIDE

- `adc_type`
    - HSADC : High-Speed ADC
    - HRADC : High-Resolution ADC
    - HSADC_PULSED : High-resolution ADC for pulsed measurements
- `meas_op_mode`
    - COMPLIANCE_SIDE
    - CURRENT
    - VOLTAGE
    - FORCE_SIDE
    - COMPLIANCE_AND_FORCE_SIDE

In [ ]:
# General Instrument Settings
# b1500.adc_averaging = 1
# b1500.adc_auto_zero = True
b1500.adc_setup('HRADC','AUTO', 6)
#b1500.adc_setup('HRADC','PLC', 1)

- `adc_setup()`
    - adc_type (`ADCType`)
    - mode (`ADCMode`)
    - N (*str, optional*)

In [ ]:
#Sweep Settings
b1500.sweep_timing(0,5,step_delay=0.1) #hold,delay
b1500.sweep_auto_abort(False,post='STOP') #disable auto abort, set post measurement output condition to stop value of sweep
# Sweep Source
nop = 11
b1500.smu1.staircase_sweep_source('VOLTAGE','LINEAR_DOUBLE','Auto Ranging',0,1,nop,0.001) #type, mode, range, start, stop, steps, compliance
# Synchronous Sweep Source
b1500.smu2.synchronous_sweep_source('VOLTAGE','Auto Ranging',0,1,0.001) #type, range, start, stop, comp
# Constant Output (could also be done using synchronous sweep source with start=stop, but then the output is not ramped up)
b1500.smu3.ramp_source('VOLTAGE','Auto Ranging',-1,stepsize=0.1,pause=20e-3) #output starts immediately! (compared to sweeps)
b1500.smu4.ramp_source('VOLTAGE','Auto Ranging',0,stepsize=0.1,pause=20e-3)

In [ ]:
#Start Measurement
b1500.check_errors()
b1500.clear_buffer()
b1500.clear_timer()
b1500.send_trigger()

In [ ]:
# read measurement data all at once
b1500.check_idle() #wait until measurement is finished
data = b1500.read_data(2*nop) #Factor 2 because of double sweep

In [ ]:
#alternatively: read measurement data live
meas = []
for i in range(nop*2):
    read_data = b1500.read_channels(4+1) # 4 measurement channels, 1 sweep source (returned due to mode=1 of data_format)
    # process live data for plotting etc.
    # data format for every channel (status code, channel name e.g. 'SMU1', data name e.g 'Current Measurement (A)', value)
    meas.append(read_data)

In [ ]:
#sweep constant sources back to 0V
b1500.smu3.ramp_source('VOLTAGE','Auto Ranging',0,stepsize=0.1,pause=20e-3)
b1500.smu4.ramp_source('VOLTAGE','Auto Ranging',0,stepsize=0.1,pause=20e-3)

#### Additional Code

In [ ]:
import pandas as pd

In [ ]:
# read measurement data all at once
b1500.check_idle()  # wait until measurement is finished
data = b1500.read_data(2 * nop)  # Factor 2 because of double sweep

In [ ]:
# Save measurement data to CSV (all at once)
df = pd.DataFrame({"Measurement Data": data})
df.to_csv("measurement_results.csv", index=False)
print("Measurement data saved to measurement_results.csv")

In [ ]:
# alternatively: read measurement data live and save to CSV
meas = []
with open("live_measurement_results.csv", "w") as f:
    f.write("Status,Channel,Data Name,Value\n")  # CSV header
    for i in range(nop * 2):
        read_data = b1500.read_channels(4 + 1)  # 4 measurement channels, 1 sweep source
        meas.append(read_data)
        
        # Save live data to CSV
        for entry in read_data:
            f.write(f"{entry[0]},{entry[1]},{entry[2]},{entry[3]}\n")
print("Live measurement data saved to live_measurement_results.csv")

### Sampling measurement with 4 SMUs

In [ ]:
# choose measurement mode
b1500.meas_mode('SAMPLING', *b1500.smu_references) #order in smu_references determines order of measurement
number_of_channels = len(b1500.smu_references)

In [ ]:
# settings for individual SMUs
for smu in b1500.smu_references:
    smu.enable() #enable SMU
    smu.adc_type = 'HSADC' #set ADC to high-speed ADC
    smu.meas_range_current = '1 nA'
    smu.meas_op_mode = 'COMPLIANCE_SIDE' # other choices: Current, Voltage, FORCE_SIDE, COMPLIANCE_AND_FORCE_SIDE

In [ ]:
b1500.sampling_mode = 'LINEAR'
# b1500.adc_averaging = 1
# b1500.adc_auto_zero = True
b1500.adc_setup('HSADC','AUTO',1)
#b1500.adc_setup('HSADC','PLC',1)
nop=11
b1500.sampling_timing(2,0.005,nop) #MT: bias hold time, sampling interval, number of points
b1500.sampling_auto_abort(False,post='BIAS') #MSC: BASE/BIAS
b1500.time_stamp = True

In [ ]:
# Sources
b1500.smu1.sampling_source('VOLTAGE','Auto Ranging',0,1,0.001) #MV/MI: type, range, base, bias, compliance
b1500.smu2.sampling_source('VOLTAGE','Auto Ranging',0,1,0.001)
b1500.smu3.ramp_source('VOLTAGE','Auto Ranging',-1,stepsize=0.1,pause=20e-3) #output starts immediately! (compared to sweeps)
b1500.smu4.ramp_source('VOLTAGE','Auto Ranging',-1,stepsize=0.1,pause=20e-3)

In [ ]:
#Start Measurement
b1500.check_errors()
b1500.clear_buffer()
b1500.clear_timer()
b1500.send_trigger()

In [ ]:
meas=[]
for i in range(nop):
    read_data = b1500.read_channels(1+2*number_of_channels) #Sampling Index + (time stamp + measurement value) * number of channels
    # process live data for plotting etc.
    # data format for every channel (status code, channel name e.g. 'SMU1', data name e.g 'Current Measurement (A)', value)
    meas.append(read_data)

In [ ]:
#sweep constant sources back to 0V
b1500.smu3.ramp_source('VOLTAGE','Auto Ranging',0,stepsize=0.1,pause=20e-3)
b1500.smu4.ramp_source('VOLTAGE','Auto Ranging',0,stepsize=0.1,pause=20e-3)